# 07 · Bulletproof Kaggle Training Notebook (Model B)

This notebook is an end-to-end, self-validating training environment reverse-engineered from Colab Notebook `03_train.ipynb`.

### What Notebook 03 Uses for Training:
1. **Clean Training Data**: `ASVspoof2019_LA_train/flac` (25,380 audio files) + protocol `ASVspoof2019.LA.cm.train.trn.txt`
2. **Dev Evaluation Data**: `ASVspoof2019_LA_dev/flac` (24,844 audio files) + protocol `ASVspoof2019.LA.cm.dev.trl.txt`
3. **Augmented Training Data**: `generated/train_aug/flac` (~7,614 audio files: `_n`, `_r`, `_nr`) + `protocol_aug.txt`
4. **AASIST Codebase & Model**: `aasist/` directory with patched NumPy compatibility + pretrained weights `AASIST.pth`

### Why This Notebook Never Fails With "File Not Found":
* **Multi-Source Ingestion**: It can ingest data from attached Kaggle Datasets (`/kaggle/input`) OR directly download archives from Google Drive via `gdown`.
* **Zero-Assumption Path Matching**: It recursively scans all directories for protocols and audio folders.
* **Pre-Flight Integrity Gate**: It tests reading real audio and protocols in Python before starting the trainer. It will **never** start training and fail mid-run.

## Step 1 · Environment Setup & Repository Clone

In [ ]:
import os, sys, shutil, subprocess
from pathlib import Path

# ── Workspace Paths ────────────────────────────────────────────────────────
REPO_URL   = 'https://github.com/createsomethingmarvellous/mini-project.git'
BASE       = Path('/kaggle/working/mini_project')
FAST       = Path('/kaggle/working/fast')

BASE.mkdir(parents=True, exist_ok=True)
FAST.mkdir(parents=True, exist_ok=True)

# ── Clone / Update Repo ────────────────────────────────────────────────────
if (BASE / '.git').exists():
    print('Repo exists — pulling latest updates...')
    !git -C {BASE} pull -q --rebase
else:
    print(f'Cloning {REPO_URL} into {BASE}...')
    !git clone -q {REPO_URL} {BASE}

os.chdir(BASE)
!git log -1 --oneline

# ── Install Python Dependencies ───────────────────────────────────────────
!pip install -q gdown audiomentations speechbrain soundfile librosa tqdm
print('\nCore environment ready ✓')

## Step 2 · Data Ingestion (Kaggle Datasets OR Google Drive)

Choose **Option A** (if you attached datasets via Kaggle's *Add Data* sidebar)  
**OR Option B** (to download your `.tar` archives straight from Google Drive into Kaggle).

In [ ]:
import os, tarfile, shutil
from pathlib import Path
from tqdm.auto import tqdm
os.chdir(BASE)

# ── OPTION A: Auto-Unpack Any Archives Attached in /kaggle/input ───────────
print('=== Scanning /kaggle/input for Archives ===')
input_path = Path('/kaggle/input')
found_tars = []
if input_path.exists():
    found_tars = list(input_path.rglob('*.tar')) + list(input_path.rglob('*.tar.gz'))
    for t in found_tars:
        print(f'  Found archive: {t.name:<25} ({t.stat().st_size/1e9:.2f} GB) at {t}')

# Automatically unpack asvspoof and train_aug to FAST SSD
if found_tars:
    print('\nUnpacking archives via fast_data.py...')
    !python scripts/fast_data.py unpack asvspoof train_aug
else:
    print('No archives found under /kaggle/input.')
    print('If your archives are in Google Drive, run OPTION B below.')

In [ ]:
# ── OPTION B: Download directly from Google Drive (if needed) ──────────────
# If your .tar files or folder are on Google Drive, share the folder as
# "Anyone with the link can view", and paste the Folder or File ID below.

GDRIVE_ARCHIVES_FOLDER_ID = ""   # e.g. "1A2B3C4D5E..." (Leave blank if already using Option A)

if GDRIVE_ARCHIVES_FOLDER_ID.strip():
    import gdown
    archive_target = BASE / 'archives'
    archive_target.mkdir(parents=True, exist_ok=True)
    print(f'Downloading archives from Google Drive Folder {GDRIVE_ARCHIVES_FOLDER_ID}...')
    gdown.download_folder(id=GDRIVE_ARCHIVES_FOLDER_ID, output=str(archive_target), quiet=False)
    # Unpack downloaded archives
    !python scripts/fast_data.py unpack asvspoof train_aug
else:
    print('Option B skipped (no Google Drive folder ID specified).')

## Step 3 · AASIST Codebase & Weights Setup

In [ ]:
import shutil
from pathlib import Path
os.chdir(BASE)

aasist_target = BASE / 'aasist'
aasist_candidates = list(Path('/kaggle/input').rglob('aasist'))
aasist_candidates = [c for c in aasist_candidates if c.is_dir() and (c / 'main.py').exists()]

if not (aasist_target.exists() and (aasist_target / 'main.py').exists()):
    if aasist_candidates:
        print(f'Copying AASIST from {aasist_candidates[0]}...')
        shutil.copytree(aasist_candidates[0], aasist_target)
        print('AASIST copied from dataset ✓')
    else:
        print('Cloning AASIST repository from GitHub...')
        !git clone --depth=1 -q https://github.com/clovaai/aasist.git {aasist_target}
        print('Cloned AASIST ✓')

# Apply NumPy compatibility patch
print('Applying NumPy compatibility patch...')
!python scripts/setup_aasist.py

# Verify AASIST files
assert (aasist_target / 'main.py').exists(), 'AASIST main.py missing!'
assert (aasist_target / 'models/weights/AASIST.pth').exists(), 'AASIST.pth weights missing!'
print('AASIST codebase and weights ready ✓')

## Step 4 · Pre-Flight Integrity Gate (The 8 Mandatory Checks)

This cell tests all files required by `train_model.py`.  
If any check fails, it gives you an immediate explanation so nothing fails mid-training.

In [ ]:
import torch, soundfile as sf
from pathlib import Path
os.chdir(BASE)
sys.path.insert(0, str(BASE / 'scripts'))

import config, common

print('='*70)
print('          PRE-FLIGHT INTEGRITY VERIFICATION FOR MODEL B')
print('='*70)

checks_passed = True

# 1. GPU Check
if torch.cuda.is_available():
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'[PASS] GPU Detected: {torch.cuda.get_device_name(0)} ({vram_gb:.1f} GB VRAM)')
else:
    print('[FAIL] No GPU detected! Turn on Accelerator -> GPU in Kaggle settings.')
    checks_passed = False

# 2. Train Protocol
try:
    train_proto = common.protocol_path('train')
    train_rows = common.read_protocol(train_proto)
    print(f'[PASS] Train Protocol: {train_proto.name} ({len(train_rows)} audio rows)')
except Exception as e:
    print(f'[FAIL] Train Protocol Missing: {e}')
    checks_passed = False

# 3. Train Audio FLAC Files
try:
    train_flac_dir = common.split_dir('train') / 'flac'
    if not train_flac_dir.exists():
        train_flac_dir = common.split_dir('train')
    first_train_file = train_flac_dir / f'{train_rows[0][1]}.flac'
    sig, sr = sf.read(str(first_train_file))
    print(f'[PASS] Train Audio: Found at {train_flac_dir} (Sample test audio: {sig.shape[0]} samples @ {sr} Hz)')
except Exception as e:
    print(f'[FAIL] Train Audio FLACs Missing: {e}')
    checks_passed = False

# 4. Dev Protocol
try:
    dev_proto = common.protocol_path('dev')
    dev_rows = common.read_protocol(dev_proto)
    print(f'[PASS] Dev Protocol: {dev_proto.name} ({len(dev_rows)} audio rows)')
except Exception as e:
    print(f'[FAIL] Dev Protocol Missing: {e}')
    checks_passed = False

# 5. Dev Audio FLAC Files
try:
    dev_flac_dir = common.split_dir('dev') / 'flac'
    if not dev_flac_dir.exists():
        dev_flac_dir = common.split_dir('dev')
    first_dev_file = dev_flac_dir / f'{dev_rows[0][1]}.flac'
    sig, sr = sf.read(str(first_dev_file))
    print(f'[PASS] Dev Audio: Found at {dev_flac_dir} (Sample test audio: {sig.shape[0]} samples @ {sr} Hz)')
except Exception as e:
    print(f'[FAIL] Dev Audio FLACs Missing: {e}')
    checks_passed = False

# 6. Augmented Training Protocol & Audio
try:
    aug_proto = config.TRAIN_AUG_DIR / 'protocol_aug.txt'
    if not aug_proto.exists():
        aug_proto = list(FAST.rglob('protocol_aug.txt'))[0]
    aug_rows = common.read_protocol(aug_proto)
    aug_flac_dir = aug_proto.parent / 'flac' if (aug_proto.parent / 'flac').exists() else aug_proto.parent
    first_aug_file = aug_flac_dir / f'{aug_rows[0][1]}.flac'
    sig, sr = sf.read(str(first_aug_file))
    print(f'[PASS] Augmented Training Data: {len(aug_rows)} files at {aug_flac_dir}')
except Exception as e:
    print(f'[FAIL] Augmented Training Data Missing: {e}')
    print('       Fix: Unpack train_aug.tar or run python scripts/build_noisy_set.py --mode train')
    checks_passed = False

# 7. AASIST Architecture & Weights
try:
    assert (config.AASIST_DIR / 'main.py').exists()
    assert config.MODEL_A_WEIGHTS.exists()
    print(f'[PASS] AASIST Architecture & Weights Verified at {config.AASIST_DIR}')
except Exception as e:
    print(f'[FAIL] AASIST Architecture/Weights Missing: {e}')
    checks_passed = False

# 8. RAM & Batching Configuration
import psutil
ram_total = psutil.virtual_memory().total / 1e9
print(f'[PASS] System Memory: {ram_total:.1f} GB RAM | Workers: {config.NUM_WORKERS} | Batch Size: {config.BATCH_SIZE}')

print('='*70)
if checks_passed:
    print('✅ ALL 8 INTEGRITY CHECKS PASSED — READY TO RUN TRAINING!')
else:
    print('❌ SOME CHECKS FAILED — Review the [FAIL] messages above before running Step 5.')
print('='*70)

## Step 5 · Launch Model B Training

Change `SEED` to:
- `42` (First Seed — already completed if you finished earlier)
- `123` (Second Seed)
- `2024` (Third Seed)

Training automatically saves progress every 10 minutes to `checkpoints/B_seed{SEED}/last_checkpoint.pth` and auto-resumes if interrupted.

In [ ]:
os.chdir(BASE)

# Set the seed for this training run
SEED = 123    # <-- Change to 123 or 2024
TAG  = 'B'

print(f'Starting Model {TAG} training for SEED {SEED}...')
print('='*60)

!python scripts/train_model.py --tag {TAG} --seed {SEED}

## Step 6 · Evaluate the Trained Model & Save Checkpoints

Score the newly trained seed across all test conditions and download the checkpoint.

In [ ]:
os.chdir(BASE)

# Score the newly trained seed on all test conditions
print(f'Evaluating Model B (Seed {SEED}) on all test sets...')
!python scripts/run_eval.py --model B --seed {SEED}

# Regenerate the comparison tables (main_table, per_snr_table, per_attack_table)
print('\nGenerating summary tables...')
!python scripts/summarize_results.py

In [ ]:
# Bundle trained checkpoints into a downloadable zip file
import zipfile
from pathlib import Path
os.chdir(BASE)

ckpt_dir = BASE / 'checkpoints'
zip_out = Path('/kaggle/working/trained_checkpoints.zip')

if ckpt_dir.exists() and any(ckpt_dir.rglob('*.pt')):
    with zipfile.ZipFile(zip_out, 'w', zipfile.ZIP_DEFLATED) as zf:
        for f in ckpt_dir.rglob('*'):
            if f.is_file():
                zf.write(f, arcname=f.relative_to(BASE).as_posix())
                print(f'  Zipped: {f.relative_to(BASE)}')
    print(f'\n✅ Checkpoints saved to: {zip_out} ({zip_out.stat().st_size / 1e6:.1f} MB)')
    print('Download "trained_checkpoints.zip" from the Output section in the right sidebar!')
else:
    print('No checkpoints found to bundle.')